In [1]:
# Packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Use your own features and labels or grab the csv files with features and labels from today's exercise.

# Load features and labels into dataframes
file_labels = "example_labels.csv"
file_features = "example_features.csv"

df = pd.read_csv("..\data\metadata_(for_scores).csv")
#features = pd.read_csv(file_features)

# Check what's inside

df.head()

# Put everything we need together
#df = df.drop(['image_id','seborrheic_keratosis'],axis=1)
#df['area'] = features['area']
#df['perimeter'] = features['perimeter']

# Please remember that area and perimeter alone are often not sufficient for classification.
# When doing your project, you will need other features.

#print(df.head())

,img_id,patient_id,lesion_id,gender,skin_cancer_diagnosis,diagnostic,biopsed,mask_components,melanoma_color_count,melanoma_colors,hue_variance,saturation_variance,value_variance,Polsby-Popper,score:
0,PAT_684_1303_368.png,PAT_684,1303,MALE,1,BCC,True,2,1,blue_gray,0.003734,0.014368,0.013598,12.927124,0
1,PAT_842_1606_971.png,PAT_842,1606,FEMALE,1,BCC,True,3,1,blue_gray,0.143109,0.001537,0.003105,13.378031,0
2,PAT_113_172_610.png,PAT_113,172,MALE,1,SCC,True,2,2,"red, blue_gray",0.002274,0.018044,0.004772,12.690194,0
3,PAT_1633_2855_460.png,PAT_1633,2855,NaN,0,ACK,False,5,0,NaN,0.001311,0.001702,0.002817,17.035902,0
4,PAT_168_262_74.png,PAT_168,262,MALE,1,BCC,True,1,1,blue_gray,0.008007,0.015162,0.003787,10.120009,0


In [2]:
df.shape

(2094, 15)

In [3]:
from sklearn.model_selection import train_test_split

x = df[['melanoma_color_count','hue_variance', 'saturation_variance', 'value_variance', 'Polsby-Popper', 'score:']]
y = df[['skin_cancer_diagnosis']]

dev_x, test_x, dev_y, test_y = train_test_split(
        x, y, stratify=y, random_state=0)

train_x, val_x, train_y, val_y = train_test_split(
        dev_x, dev_y, stratify=dev_y, random_state=0)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=True, with_std=True)

# Fit på træningsdata
scaler.fit(train_x)

# Transformér både train og test area
train_x_scaled = scaler.transform(train_x)
test_x_scaled = scaler.transform(test_x)
val_x_scaled = scaler.transform(val_x)

In [5]:
n_neighbors = [1, 3, 5, 7, 9]

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
auc_train = []
auc_val = []

for n in n_neighbors:
    knn1 = KNeighborsClassifier(n_neighbors=n)
    knn1_trained = knn1.fit(train_x_scaled, np.ravel(train_y))
    train_pred_knn1 = knn1_trained.predict(train_x_scaled)
    val_pred_knn1 = knn1_trained.predict(val_x_scaled)

    auc_train.append(roc_auc_score(train_y, train_pred_knn1))
    auc_val.append(roc_auc_score(val_y, val_pred_knn1))

In [7]:
auc_train

[np.float64(1.0),
 np.float64(0.7935427891639669),
 np.float64(0.7392745563428679),
 np.float64(0.7179651542188573),
 np.float64(0.6992660094823953)]

In [8]:
auc_val

[np.float64(0.5929115300942711),
 np.float64(0.6157023723194862),
 np.float64(0.590269864290894),
 np.float64(0.6310732414793329),
 np.float64(0.6362011809800062)]